In [1]:
import sys

import pm4py

import pandas as pd
import numpy as np

import torch
from torch.utils.data import DataLoader

from sklearn.model_selection import train_test_split

from utils.general_utils import set_stdout_to_file, set_seed

from config.feature_config import FeatureConfig

from model.preprocessor import PreprocessorArtifacts
from model.next_event_model import ProcessLSTM, train_ProcessLSTM, validate_ProcessLSTM

### --- Preprocess dataset ---

In [2]:
set_seed(seed=42)

In [3]:
log = pm4py.read_xes("../../data/Sepsis Cases - Event Log.xes")

C:\Users\dcoralage\Downloads\counterfactual_prediction_experiments\counterfactual_env\lib\site-packages\pm4py\utils.py:1027: UserWarning: Install the optional requirement `r4pm` to import/export files faster. `rustxes` remains supported as a fallback.
  warnings.warn(
C:\Users\dcoralage\Downloads\counterfactual_prediction_experiments\counterfactual_env\lib\site-packages\pm4py\util\dt_parsing\parser.py:82: UserWarning: ISO8601 strings are not fully supported with strpfromiso for Python versions below 3.11
  warnings.warn(


parsing log, completed traces ::   0%|          | 0/1050 [00:00<?, ?it/s]

In [4]:
df = pm4py.convert_to_dataframe(log)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15214 entries, 0 to 15213
Data columns (total 32 columns):
 #   Column                     Non-Null Count  Dtype              
---  ------                     --------------  -----              
 0   InfectionSuspected         1050 non-null   object             
 1   org:group                  15214 non-null  object             
 2   DiagnosticBlood            1050 non-null   object             
 3   DisfuncOrg                 1050 non-null   object             
 4   SIRSCritTachypnea          1050 non-null   object             
 5   Hypotensie                 1050 non-null   object             
 6   SIRSCritHeartRate          1050 non-null   object             
 7   Infusion                   1050 non-null   object             
 8   DiagnosticArtAstrup        1050 non-null   object             
 9   concept:name               15214 non-null  object             
 10  Age                        1050 non-null   float64            
 11  Di

In [6]:
df.isnull().any()

InfectionSuspected            True
org:group                    False
DiagnosticBlood               True
DisfuncOrg                    True
SIRSCritTachypnea             True
Hypotensie                    True
SIRSCritHeartRate             True
Infusion                      True
DiagnosticArtAstrup           True
concept:name                 False
Age                           True
DiagnosticIC                  True
DiagnosticSputum              True
DiagnosticLiquor              True
DiagnosticOther               True
SIRSCriteria2OrMore           True
DiagnosticXthorax             True
SIRSCritTemperature           True
time:timestamp               False
DiagnosticUrinaryCulture      True
SIRSCritLeucos                True
Oligurie                      True
DiagnosticLacticAcid          True
lifecycle:transition         False
Diagnose                      True
Hypoxie                       True
DiagnosticUrinarySediment     True
DiagnosticECG                 True
case:concept:name   

In [7]:
df = df.rename(columns={"Age": "case:age"})

df["case:age"] = (
    df.groupby("case:concept:name")["case:age"]
      .transform(lambda s: s.dropna().iloc[0] if s.notna().any() else pd.NA)
)

In [8]:
event_cols = [
    "InfectionSuspected",
    "org:group",
    "DiagnosticBlood",
    "DisfuncOrg",
    "SIRSCritTachypnea",
    "Hypotensie",
    "SIRSCritHeartRate",
    "Infusion",
    "DiagnosticArtAstrup",
    "DiagnosticIC",
    "DiagnosticSputum",
    "DiagnosticLiquor",
    "DiagnosticOther",
    "SIRSCriteria2OrMore",
    "DiagnosticXthorax",
    "SIRSCritTemperature",
    "DiagnosticUrinaryCulture",
    "SIRSCritLeucos",
    "Oligurie",
    "DiagnosticLacticAcid",
    "Diagnose",
    "Hypoxie",
    "DiagnosticUrinarySediment",
    "DiagnosticECG",
    "Leucocytes",
    "CRP",
    "LacticAcid",
]

delete_cols = []

for col in event_cols:
    s = df[col].replace("", pd.NA)

    missing = s.isna().mean()
    max_freq = s.value_counts(normalize=True, dropna=True).max()

    print(
        f"{col:15} "
        f"missing={missing:.1%} "
        f"largest_class={max_freq:.1%}"
    )

    if missing >= 0.90:
        delete_cols.append(col)

print("\nColumns to delete:")
print(delete_cols)

InfectionSuspected missing=93.1% largest_class=80.8%
org:group       missing=0.0% largest_class=53.3%
DiagnosticBlood missing=93.1% largest_class=78.4%
DisfuncOrg      missing=93.1% largest_class=93.4%
SIRSCritTachypnea missing=93.1% largest_class=57.2%
Hypotensie      missing=93.1% largest_class=94.9%
SIRSCritHeartRate missing=93.1% largest_class=77.2%
Infusion        missing=93.1% largest_class=75.8%
DiagnosticArtAstrup missing=93.1% largest_class=71.7%
DiagnosticIC    missing=93.1% largest_class=80.8%
DiagnosticSputum missing=93.1% largest_class=97.2%
DiagnosticLiquor missing=93.1% largest_class=99.5%
DiagnosticOther missing=93.1% largest_class=99.0%
SIRSCriteria2OrMore missing=93.1% largest_class=81.2%
DiagnosticXthorax missing=93.1% largest_class=74.8%
SIRSCritTemperature missing=93.1% largest_class=76.3%
DiagnosticUrinaryCulture missing=93.1% largest_class=54.7%
SIRSCritLeucos  missing=93.1% largest_class=95.3%
Oligurie        missing=93.1% largest_class=97.6%
DiagnosticLacticAci

In [9]:
df = df.drop(columns=['InfectionSuspected', 'DiagnosticBlood', 'DisfuncOrg', 'SIRSCritTachypnea', 'Hypotensie', 'SIRSCritHeartRate', 
                      'Infusion', 'DiagnosticArtAstrup', 'DiagnosticIC', 'DiagnosticSputum', 'DiagnosticLiquor', 'DiagnosticOther', 
                      'SIRSCriteria2OrMore', 'DiagnosticXthorax', 'SIRSCritTemperature', 'DiagnosticUrinaryCulture', 'SIRSCritLeucos', 
                      'Oligurie', 'DiagnosticLacticAcid', 'Diagnose', 'Hypoxie', 'DiagnosticUrinarySediment', 'DiagnosticECG', 'LacticAcid'])

In [10]:
df['case:concept:name'] = df['case:concept:name'].astype('string')
df['concept:name'] = df['concept:name'].astype('string')
df['lifecycle:transition'] = df['lifecycle:transition'].astype('string')

df['time:timestamp'] = pd.to_datetime(df['time:timestamp'], errors='coerce')

df['org:group'] = df['org:group'].astype('string')

df['case:age'] = df['case:age'].astype(np.float32)
df['Leucocytes'] = df['Leucocytes'].astype(np.float32)
df['CRP'] = df['CRP'].astype(np.float32)

In [11]:
df = df.sort_values(by=['case:concept:name', 'time:timestamp'], ascending=[True, True])

In [12]:
df['time_delta'] = df.groupby('case:concept:name')['time:timestamp'].diff()
df['time_delta'] = df['time_delta'].dt.total_seconds()
df['time_delta'] = df['time_delta'].fillna(0)

In [13]:
exclude_cols = ["case:concept:name", "time:timestamp"]

sorted_cols = sorted(
    [c for c in df.columns if c not in exclude_cols]
)

df = df[exclude_cols + sorted_cols]

In [14]:
# Remove activities only found in validation data
df = df[df['org:group'] != 'X']
df = df[df['org:group'] != 'Y']
df = df[df['org:group'] != '?']

In [15]:
df.head(20)

,case:concept:name,time:timestamp,CRP,Leucocytes,case:age,concept:name,lifecycle:transition,org:group,time_delta
0,A,2014-10-22 11:15:41+00:00,NaN,NaN,85.0,ER Registration,complete,A,0.0
1,A,2014-10-22 11:27:00+00:00,NaN,9.6,85.0,Leucocytes,complete,B,679.0
2,A,2014-10-22 11:27:00+00:00,21.0,NaN,85.0,CRP,complete,B,0.0
3,A,2014-10-22 11:27:00+00:00,NaN,NaN,85.0,LacticAcid,complete,B,0.0
4,A,2014-10-22 11:33:37+00:00,NaN,NaN,85.0,ER Triage,complete,C,397.0
5,A,2014-10-22 11:34:00+00:00,NaN,NaN,85.0,ER Sepsis Triage,complete,A,23.0
6,A,2014-10-22 14:03:47+00:00,NaN,NaN,85.0,IV Liquid,complete,A,8987.0
7,A,2014-10-22 14:03:47+00:00,NaN,NaN,85.0,IV Antibiotics,complete,A,0.0
8,A,2014-10-22 14:13:19+00:00,NaN,NaN,85.0,Admission NC,complete,D,572.0
9,A,2014-10-24 09:00:00+00:00,109.0,NaN,85.0,CRP,complete,B,154001.0


In [16]:
num_cases = df['case:concept:name'].nunique()
print(f"Total number of unique cases: {num_cases}")

Total number of unique cases: 1050


### --- Feature Configurations ---

In [17]:
# --- Define feature specs ---
feature_specs = {

    "org:group": {
        "type":            "categorical",
        "level":           "event",
        "vary":            True,
    },

    "case:age": {
        "type":            "continuous",
        "level":           "case",
        "vary":            True,
    },
   
    "time_delta": {
        "type":            "continuous",
        "level":           "event",
        "vary":            True,
        "quantile_low":    0.20,
        "quantile_high":   0.80, 
    },

    "Leucocytes": {
        "type":            "continuous",
        "level":           "event",
        "vary":            True,
        "quantile_low":    0.05,
        "quantile_high":   0.90, 
    },

     "CRP": {
        "type":            "continuous",
        "level":           "event",
        "vary":            True,
    },

    # immutable
    "concept:name": {
        "type":            "categorical", 
        "level":           "event",
        "vary":            False
    },

    "lifecycle:transition": {
        "type":           "categorical", 
        "level":          "event",
        "vary":           False
    },
}

In [18]:
feature_config = FeatureConfig.from_dataframe(
    df=df,
    feature_specs=feature_specs,
    activity_feature="concept:name",
    is_robust=True,
    default_quantile_low=0.05,
    default_quantile_high=0.95
)

feature_config.save()

In [19]:
# feature_config = FeatureConfig.load()

In [20]:
feature_config.summary()

  activity_feature:    concept:name
  feature_order:       ['CRP', 'Leucocytes', 'case:age', 'concept:name', 'lifecycle:transition', 'org:group', 'time_delta']
--------------------------------------------------------------------------------------------------------------------------------------
Feature                        Type           Level    Vary   Range/Categories                         MAD        Source              
--------------------------------------------------------------------------------------------------------------------------------------
org:group                      categorical    event    yes    ['A', 'B', 'C', ...]                     N/A        data_derived        
case:age                       continuous     case     yes    [40.00, 90.00]                           10.0000    quantile_derived    
time_delta                     continuous     event    yes    [0.00, 38748.80]                         139.0000   quantile_derived    
Leucocytes                    

### --- Next event prediction model ---

In [21]:
cat_cols = df.select_dtypes(include=["string"]).columns
for col in cat_cols:
    df[col] = df[col].fillna("NA").astype('string')

In [22]:
case_ids = df["case:concept:name"].unique()

train_cases, val_cases = train_test_split(case_ids, test_size=0.2, random_state=42)

train_df = df[df["case:concept:name"].isin(train_cases)].copy()
val_df   = df[df["case:concept:name"].isin(val_cases)].copy()

In [23]:
preprocessor_artifacts = PreprocessorArtifacts.build(
    df=train_df,
    feature_config=feature_config,      
    scaler_type="robust",
)

preprocessor_artifacts.save()

In [24]:
# preprocessor_artifacts = PreprocessorArtifacts.load()

In [25]:
preprocessor_artifacts.summary()

===================PreprocessorArtifacts====================
  scaler:              RobustScaler
  encoders:            ['org:group', 'concept:name', 'lifecycle:transition']
  activity_prototypes: 15 activities


In [26]:
# Transform nan cols to 0
float_cols = df.select_dtypes(include=["float32", "float64"]).columns
df[float_cols] = df[float_cols].fillna(0)
train_df[float_cols] = train_df[float_cols].fillna(0)
val_df[float_cols] = val_df[float_cols].fillna(0)

In [27]:
train_dataset = preprocessor_artifacts.transform_dataframe_to_processdataset(
    case_id_field="case:concept:name", 
    df=train_df,
    sort_field="time:timestamp"
)

val_dataset = preprocessor_artifacts.transform_dataframe_to_processdataset(
    case_id_field="case:concept:name", 
    df=val_df,
    sort_field="time:timestamp"
)

In [28]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [29]:
print(preprocessor_artifacts.get_categorical_feature_cardinality())

{'dynamic_categorical_info': {'concept:name': 15, 'lifecycle:transition': 1, 'org:group': 23}, 'static_categorical_info': {}}


In [30]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [31]:
device

device(type='cuda')

In [32]:
criterion = torch.nn.CrossEntropyLoss()

In [33]:
log_file, original_stdout = set_stdout_to_file(filepath="logs/sepsis-model_output.txt")

Epoch 020/100 | Train Loss: 0.8130 | LR: 9.05e-04
Epoch 040/100 | Train Loss: 0.6669 | LR: 6.55e-04
Epoch 060/100 | Train Loss: 0.5314 | LR: 3.46e-04
Epoch 080/100 | Train Loss: 0.4574 | LR: 9.64e-05
Epoch 100/100 | Train Loss: 0.4367 | LR: 1.00e-06
Time taken for next event model (training): 149.297378 seconds
Time taken for next event model (validation): 0.095998 seconds
Val loss: {'loss': 1.31485273740692, 'accuracy': 0.6517719568567026, 'f1_macro': 0.4655175575960256, 'f1_weighted': 0.6455596733912354}


In [34]:
embedding_metadata = preprocessor_artifacts.get_embedding_metadata()

model = ProcessLSTM(
    dynamic_categorical_info=embedding_metadata["dynamic_categorical_info"],
    static_categorical_info=embedding_metadata["static_categorical_info"],
    n_dynamic_continuous=embedding_metadata["n_dynamic_continuous"],
    n_static_continuous=embedding_metadata["n_static_continuous"],
    n_classes=embedding_metadata["n_classes"]
)

train_loss_history = train_ProcessLSTM(
    model=model,
    train_loader=train_loader,
    learning_rate=1e-3,
    criterion=criterion,
    num_epochs=100,
    device=device
)

model.save()

In [35]:
# model = ProcessLSTM.load()

In [36]:
val_loss = validate_ProcessLSTM(
    model=model,
    val_loader=val_loader,
    criterion=criterion,
    device=device
)

print("Val loss:", val_loss)

### --- Cleanup ---

In [37]:
if df["time:timestamp"].dt.tz is not None:
    df["time:timestamp"] = df["time:timestamp"].dt.tz_convert(None)
df.to_excel("../../data/sepsis.xlsx", index=False, engine="openpyxl")

In [38]:
sys.stdout = original_stdout
log_file.close()